## Source-only notebook copy

All saved outputs, embedded media/attachments, execution counts and environment-specific metadata have been removed from this copy. Historical results and limitations are documented in README.md; the original academic submission is unchanged. No project cells were executed during packaging. Input images, datasets, model weights and historical result files are not included.


# **Laboratory Activity 2: Can a Simple Neural Network Recognize Images?**

## Part 1 — Dataset Exploration




### Download the dataset

In [ ]:
import os
import kagglehub

download_path = kagglehub.dataset_download("crowww/a-large-scale-fish-dataset")
print("Downloaded to:", download_path)

fish_root = None
for dirpath, dirnames, filenames in os.walk(download_path):
    if os.path.basename(dirpath) == "Fish_Dataset":
        fish_root = dirpath
        break

print("Using Fish_Dataset folder at:", fish_root)

### Build a clean directory without the unnecessary files/folders

In [ ]:
import shutil

CLEAN_DIR = "/content/fish_dataset_clean"
IMAGE_EXTS = (".png", ".jpg", ".jpeg")

if os.path.exists(CLEAN_DIR):
    shutil.rmtree(CLEAN_DIR)
os.makedirs(CLEAN_DIR, exist_ok=True)

n_copied = 0
n_skipped_gt = 0

for dirpath, dirnames, filenames in os.walk(fish_root):
    folder_label = os.path.basename(dirpath)
    if "GT" in folder_label:
        n_skipped_gt += len(filenames)
        continue
    for fname in filenames:
        if fname.lower().endswith(IMAGE_EXTS):
            dest_dir = os.path.join(CLEAN_DIR, folder_label)
            os.makedirs(dest_dir, exist_ok=True)
            shutil.copy(os.path.join(dirpath, fname), os.path.join(dest_dir, fname))
            n_copied += 1

### Dataset characteristics

In [ ]:
import glob
from PIL import Image
import pandas as pd
import tensorflow as tf

dataset = tf.keras.preprocessing.image_dataset_from_directory(CLEAN_DIR)
class_names = dataset.class_names
NUM_CLASSES = len(class_names)

records = []
for cls in class_names:
    for fp in glob.glob(os.path.join(CLEAN_DIR, cls, "*")):
        records.append({"filepath": fp, "label": cls})
df = pd.DataFrame(records)

class_counts = df["label"].value_counts()
with Image.open(df.iloc[0]["filepath"]) as im:
    sample_format, sample_size, sample_mode = im.format, im.size, im.mode

print(f"Number of Classes: {NUM_CLASSES}")
print("Classes:", class_names)
print(f"Total Number of Images: {len(df)}")
print(f"Image Format: {sample_format}")
print(f"Image Dimensions (sample): {sample_size}")
print(f"Color Channels: {sample_mode}")
print(f"Smallest Class: {class_counts.idxmin()} ({class_counts.min()} images)")
print(f"Largest Class: {class_counts.idxmax()} ({class_counts.max()} images)")

classes_table = class_counts.reset_index()
classes_table.columns = ["Class Name", "Number of Images"]
classes_table.index = classes_table.index + 1
classes_table

### Sample images from each class

In [ ]:
# Display the 1 sample fish image for each class
import matplotlib.pyplot as plt

seen = {}
for images, labels in dataset.unbatch():
    label = int(labels.numpy())
    if label not in seen:
        seen[label] = images.numpy().astype("uint8")
    if len(seen) == NUM_CLASSES:
        break

plt.figure(figsize=(10, 10))
for i, label in enumerate(sorted(seen)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(seen[label])
    plt.title(class_names[label])
    plt.axis("off")
plt.tight_layout()

## Part 2 — Train, Validation, and Test Sets

In [ ]:
ds_train = tf.keras.preprocessing.image_dataset_from_directory(
    CLEAN_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
)
ds_validation = tf.keras.preprocessing.image_dataset_from_directory(
    CLEAN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
)

In [ ]:
# Create a test set out of part of the validation split
val_batches = tf.data.experimental.cardinality(ds_validation)
test_dataset = ds_validation.take(val_batches // 2)
validation_dataset = ds_validation.skip(val_batches // 2)

def count_images(tf_dataset):
    return sum(labels.shape[0] for _, labels in tf_dataset)

n_train = count_images(ds_train)
n_val = count_images(validation_dataset)
n_test = count_images(test_dataset)
n_total = n_train + n_val + n_test

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Testing", "Total"],
    "Number of Images": [n_train, n_val, n_test, n_total],
})
split_summary["Percentage"] = (split_summary["Number of Images"] / n_total * 100).round(2)
split_summary

## Part 3 — Simple Neural Network

In [ ]:
BASELINE_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 15

def resize_ds(tf_dataset, size):
    return tf_dataset.map(lambda image, label: (tf.image.resize(image, (size, size)) / 255.0, label))

ds_train_r = resize_ds(ds_train, BASELINE_SIZE)
ds_val_r = resize_ds(validation_dataset, BASELINE_SIZE)
ds_test_r = resize_ds(test_dataset, BASELINE_SIZE)

In [ ]:
# One-hot encode labels (same pattern as the original notebook)
def input_preprocess(image, label):
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

ds_train_r = ds_train_r.map(input_preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
ds_val_r = ds_val_r.map(input_preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
ds_test_r = ds_test_r.map(input_preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [ ]:
def build_simple_nn(img_size, num_classes):
    return tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(img_size, img_size, 3)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])

model = build_simple_nn(BASELINE_SIZE, NUM_CLASSES)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## Part 4 — Model Training

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor="val_accuracy", min_delta=0.01, patience=10, verbose=1)

history = model.fit(
    ds_train_r,
    epochs=EPOCHS,
    validation_data=ds_val_r,
    callbacks=[es],
    verbose=1,
)

In [ ]:
# Figure 3 — Training and Validation Loss
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Figure 3. Training and Validation Loss")
plt.ylabel("loss")
plt.xlabel("epoch")
plt.legend(["train", "validation"], loc="upper left")
plt.show()

In [ ]:
# Figure 4 — Training and Validation Accuracy
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Figure 4. Training and Validation Accuracy")
plt.ylabel("accuracy")
plt.xlabel("epoch")
plt.legend(["train", "validation"], loc="upper left")
plt.show()

## Part 5 — Experiment: Image Resolution

In [ ]:
import time
import random
import numpy as np

resolutions = [32, 64, 128]
results = []
histories = {}
trained_models = {}

for size in resolutions:
    print(f"\n=== Experiment: {size}x{size} ===")

    SEED = 123
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    tr = resize_ds(ds_train, size).map(input_preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    va = resize_ds(validation_dataset, size).map(input_preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    te = resize_ds(test_dataset, size).map(input_preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    exp_model = build_simple_nn(size, NUM_CLASSES)
    exp_model.compile(
        optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
    )

    input_features = size * size * 3
    n_params = exp_model.count_params()

    start = time.time()
    hist = exp_model.fit(tr, epochs=EPOCHS, validation_data=va, verbose=0)
    elapsed = time.time() - start

    train_acc = hist.history["accuracy"][-1]
    val_acc = hist.history["val_accuracy"][-1]
    test_loss, test_acc = exp_model.evaluate(te, verbose=0)

    results.append({
        "Resolution": f"{size}x{size}",
        "Input Features": input_features,
        "Parameters": f"{n_params:,}",
        "Training Accuracy": round(train_acc, 4),
        "Validation Accuracy": round(val_acc, 4),
        "Test Accuracy": round(test_acc, 4),
        "Training Time (s)": round(elapsed, 1),
    })
    
    histories[size] = hist
    trained_models[size] = exp_model

    print(
        f"Params: {n_params:,} | Train acc: {train_acc:.4f} | "
        f"Val acc: {val_acc:.4f} | Test acc: {test_acc:.4f} | Time: {elapsed:.1f}s"
    )

results_df = pd.DataFrame(results)
results_df

## Part 6 — Experiment Results

In [ ]:
# Figure 5 — Test Accuracy by Image Resolution
plt.figure(figsize=(6, 4))
plt.bar(results_df["Resolution"], results_df["Test Accuracy"], color="steelblue")
plt.title("Figure 5. Test Accuracy by Image Resolution")
plt.xlabel("Resolution"); plt.ylabel("Test Accuracy")
plt.ylim(0, 1)
plt.show()

In [ ]:
# Figure 6 — Training Time by Image Resolution
plt.figure(figsize=(6, 4))
plt.bar(results_df["Resolution"], results_df["Training Time (s)"], color="indianred")
plt.title("Figure 6. Training Time by Image Resolution")
plt.xlabel("Resolution"); plt.ylabel("Training Time (seconds)")
plt.show()

## Part 7 — Model Evaluation

In [ ]:
final_model = trained_models[32]
final_size = 32
final_test_ds = resize_ds(test_dataset, final_size).map(input_preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

test_loss, test_acc = final_model.evaluate(final_test_ds, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Collect true and predicted labels from the tf.data test set
y_true, y_pred = [], []
for images, labels in final_test_ds:
    preds = final_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Figure 7 — Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Figure 7. Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 7.3 — Classification report
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
report_df = pd.DataFrame(report).transpose().round(3)
report_df

In [ ]:
# 7.4 — Best / most difficult class (by F1-score)
per_class = report_df.loc[class_names]
best_class = per_class["f1-score"].idxmax()
worst_class = per_class["f1-score"].idxmin()
print("Best performing class:", best_class, "-> F1:", per_class.loc[best_class, "f1-score"])
print("Most difficult class:", worst_class, "-> F1:", per_class.loc[worst_class, "f1-score"])

## Part 8 — Prediction Analysis

In [ ]:
test_images_r = resize_ds(test_dataset, final_size)
X_list, y_list = [], []
for images, labels in test_images_r:
    X_list.append(images.numpy())
    y_list.append(labels.numpy())
X_test_vis = np.concatenate(X_list, axis=0)
y_test_vis = np.concatenate(y_list, axis=0)

correct_idx = np.where(y_pred == y_true)[0]
incorrect_idx = np.where(y_pred != y_true)[0]

def show_predictions(indices, title, n=9):
    n = min(n, len(indices))
    chosen = np.random.choice(indices, size=n, replace=False)
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    for ax, idx in zip(axes.flatten(), chosen):
        ax.imshow(X_test_vis[idx])
        ax.set_title(f"Actual: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=8)
        ax.axis("off")
    for ax in axes.flatten()[n:]:
        ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Figure 8 — Correct predictions
show_predictions(correct_idx, "Figure 8. Correct Predictions")

In [ ]:
# Figure 9 — Incorrect predictions
show_predictions(incorrect_idx, "Figure 9. Incorrect Predictions")

In [ ]:
# 8.3 — Table of a few incorrect predictions for manual analysis
sample_errors = np.random.choice(incorrect_idx, size=min(5, len(incorrect_idx)), replace=False)
errors_df = pd.DataFrame({
    "Index": sample_errors,
    "Actual Class": [class_names[y_true[i]] for i in sample_errors],
    "Predicted Class": [class_names[y_pred[i]] for i in sample_errors],
})
errors_df